# Lesson 3: Memory
So far, every `invoke` started from nothing. The agent forgot everything between calls. A real user says *"I have $5,000"* once and expects the agent to remember. This lesson fixes that.

Run the cells top to bottom with **Shift+Enter**.

## Step 0: Setup
The tools now live in `stock_advisor/tools.py`, so we import them instead of copying them.

In [1]:
import sys
sys.path.insert(0, '..')   # lets the notebook find the stock_advisor folder

from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from stock_advisor.tools import ALL_TOOLS

load_dotenv()
MODEL = 'groq:openai/gpt-oss-120b'

## Step 1: Prove the problem
This agent has no memory. We tell it a fact, then ask for it back in a new call.

In [2]:
forgetful = create_agent(model=MODEL, tools=[], system_prompt='You are a helpful assistant. Be brief.')

forgetful.invoke({'messages': [{'role': 'user', 'content': 'My budget is $5,000.'}]})
reply = forgetful.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]})
print(reply['messages'][-1].content)

I don’t have any information about your budget. Could you provide more details or let me know what you’re planning?


It cannot know. The second call never saw the first. The model has no memory of its own. **We** have to send it the earlier messages every time.

## Step 2: Add memory with a checkpointer
A *checkpointer* saves the conversation after every step. A *thread id* is the name of one conversation. Same thread id means same conversation.

In [3]:
agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt='You are a helpful assistant. Be brief.',
    checkpointer=InMemorySaver(),   # this one line adds memory
)

alice = {'configurable': {'thread_id': 'alice'}}

agent.invoke({'messages': [{'role': 'user', 'content': 'My budget is $5,000.'}]}, config=alice)
reply = agent.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]}, config=alice)
print(reply['messages'][-1].content)

Your budget is $5,000.


## Step 3: Different thread, different conversation
Each user gets their own thread id. Their conversations never mix.

In [4]:
bob = {'configurable': {'thread_id': 'bob'}}

reply = agent.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]}, config=bob)
print(reply['messages'][-1].content)

I’m not sure—could you share the details of your budget or what you’re planning?


## Step 4: Look at what was saved
You can read a thread's saved messages at any time.

In [5]:
state = agent.get_state(alice)
for m in state.values['messages']:
    print(m.type.upper(), ':', m.content)

HUMAN : My budget is $5,000.
AI : Got it! What are you looking to spend the $5,000 on—travel, a home project, a special event, or something else?
HUMAN : What is my budget?
AI : Your budget is $5,000.


## Step 5: The stock advisor with memory
Now the real tools plus memory. We state a budget and a style once, then ask follow-up questions without repeating ourselves.

In [6]:
from IPython.display import display, Markdown

advisor = create_agent(
    model=MODEL,
    tools=ALL_TOOLS,
    system_prompt=(
        "You are a careful stock research assistant. The user gives company names, not tickers. "
        "Find the ticker first, then use the tools for evidence.\n"
        "Rules: only state numbers that came from a tool; say 'I don't have data' instead of guessing; "
        "remember what the user told you earlier (budget, risk level); "
        "keep answers under 250 words and do not add extra sections the user did not ask for; "
        "always say this is research, not financial advice."
    ),
    checkpointer=InMemorySaver(),
)

me = {'configurable': {'thread_id': 'me'}}

def ask(text):
    result = advisor.invoke({'messages': [{'role': 'user', 'content': text}]}, config=me)
    display(Markdown(result['messages'][-1].content))   # renders tables and bold text, and never truncates

ask("I have a budget of $5,000 and I prefer low risk. I'm thinking about Apple.")

**Apple Inc. (AAPL) – Quick Research Summary**

| Item | Value (from data) |
|------|-------------------|
| **Ticker** | AAPL |
| **Sector** | Technology |
| **Current Price** | **$354.07** |
| **Trailing P/E** | **38.9** |
| **Forward P/E** | **33.5** |
| **Market Capitalization** | **$5.15 trillion** |
| **52‑Week Range** | **$243.42 – $360.83** |
| **6‑Month Price Performance** | **+15.0 %** (from $307.83 to $354.07) |
| **6‑Month High / Low** | **$356.27 / $274.70** |
| **S&P 500 6‑Month Performance (Benchmark)** | **‑4.2 %** (start $7,519.12 → end $7,201.09) |
| **Analyst Recommendation (summary)** | **Buy** |

### How Apple fits a **low‑risk, $5,000 budget**  

| Consideration | Why it matters for low‑risk investors |
|---------------|----------------------------------------|
| **Size & Stability** | Apple is a mega‑cap company (> $5 trillion market cap) with strong cash flows and a diversified product ecosystem. Large‑cap stocks tend to be less volatile than small‑cap or specula

In [7]:
ask('How many shares could I afford, and does that fit my risk level?')

**1. How many Apple shares fit a $5,000 budget?**

- **Current price (AAPL):** $354.07 per share  
- **Maximum whole shares you can buy:** `$5,000 ÷ $354.07 ≈ 14.13` → **14 shares**  

| Shares | Cost (14 × $354.07) |
|--------|---------------------|
| **14** | **$4,956.98** |

You would have about **$43** of cash left over (or you could keep it as a small buffer for transaction fees, taxes, or future purchases).

---

**2. Does a 14‑share Apple position match a “low‑risk” profile?**

| Factor | What the data shows | Implication for low‑risk investors |
|--------|---------------------|------------------------------------|
| **Company size & stability** | Apple is a mega‑cap ($5.15 trillion market cap) with strong cash flow and a diversified product & services ecosystem. | Large‑cap status generally reduces volatility compared with small‑cap or speculative stocks. |
| **Recent performance vs. market** | Apple +15 % over the last 6 months while the S&P 500 –4.2 % (Apple outperformed a fa

In [8]:
ask('What about Tesla instead? Compare it with what we just looked at.')

**Tesla, Inc. (TSLA) – Quick Research Summary**

| Item | Value (from data) |
|------|-------------------|
| **Ticker** | TSLA |
| **Sector** | Consumer Cyclical |
| **Current Price** | **$384.68** |
| **Trailing P/E** | **422.7** |
| **Forward P/E** | **178.2** |
| **Market Capitalization** | **$1.52 trillion** |
| **52‑Week Range** | **$297.38 – $498.83** |
| **6‑Month Price Performance** | **‑11.3 %** (from $433.59 to $384.68) |
| **6‑Month High / Low** | **$442.10 / $298.32** |
| **Analyst Recommendation (summary)** | **Buy** |

---

## 1. How many Tesla shares can you buy with $5,000?

- **Current price (TSLA):** $384.68 per share  
- **Maximum whole shares:** `$5,000 ÷ $384.68 ≈ 12.99` → **12 shares**  

| Shares | Cost (12 × $384.68) |
|--------|---------------------|
| **12** | **$4,616.16** |

You would have about **$383.84** of cash left over (usable for fees, a small buffer, or a tiny additional position).

---

## 2. Apple vs. Tesla – Risk‑Fit Comparison (for a low‑risk inv

## Important limit
`InMemorySaver` keeps memory **in the notebook's memory only**. If you restart the kernel, everything is gone. A real app uses a saver that writes to a database (LangGraph has SQLite and Postgres versions). The code is the same: only the checkpointer line changes.

**Another limit:** long conversations get expensive and eventually do not fit in the model's context window. We will deal with that in Lesson 6.

## Exercise
1. In Step 5, ask *"What is my budget?"* in a **new** thread id. Does it know?
2. Run `advisor.get_state(me)` and count how many messages the thread holds after three questions. Why so many?
3. Restart the kernel (Restart button at the top), run Step 5 again, and ask the budget question. What happened to the memory?

In [ ]:
# your code here